# Review: The PyTorch Workflow on a Classification Problem

Last week, we:
* built models **by hand** (Notebook 1): parameters as tensors, our own loss functions, our own update rule
* replaced those pieces with **PyTorch's tools** (Notebook 2): `nn.Linear`, `nn.MSELoss`, `torch.optim.SGD`

This notebook reviews that workflow on a new problem, **classification**, and adds the remaining tools we need before working with images:

1. **Review**: tensors, standardizing with broadcasting, and the five-step training loop, now for logistic regression
2. **Mini-batches** with `Dataset` and `DataLoader`
3. **Reusable training and evaluation functions**
4. **A neural network** with `nn.Sequential`, and better optimizers (momentum, Adam)
5. **Writing a model as a class** with `nn.Module`
6. **From 2 classes to 10**: what changes for the handwritten digits in the next notebook

The data is the breast cancer dataset from Notebook 1: 569 tumors, 30 measurements each, labeled malignant (0) or benign (1).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

As in Notebook 2: choose the device once, at the top, and use the `device` variable everywhere.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

# Part 1: Review: data, tensors, and the five-step loop

## The data

We split into training, validation, and test sets, as in Notebook 2:
* **training**: to fit the parameters
* **validation**: to compare models and settings
* **test**: used once, at the end

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42)

print(X_train.shape, X_valid.shape, X_test.shape)
print("fraction benign (label 1):", y_train.mean().round(3))

## Tensors and standardizing

Review questions, before running the next cell:
* NumPy gives 64-bit floats. What `dtype` do PyTorch layers expect?
* `X_train.mean(dim=0)` has what shape? How does broadcasting use it to standardize every column?
* `nn.Linear(30, 1)` outputs shape `(n, 1)`. What shape should the labels have?

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)

means = X_train.mean(dim=0)          # shape (30,): one mean per feature
stds = X_train.std(dim=0)
X_train = (X_train - means) / stds   # (n, 30) - (30,) broadcasts across the rows
X_valid = (X_valid - means) / stds   # always use the TRAINING statistics
X_test  = (X_test  - means) / stds

# Labels: floats (the loss needs floats), shaped (n, 1) to match the model's output
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_valid = torch.tensor(y_valid, dtype=torch.float32).reshape(-1, 1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).reshape(-1, 1)

n_features = X_train.shape[1]
print(means.shape, X_train.shape, y_train.shape)

## Logistic regression with PyTorch's tools

In Notebook 1 we wrote logistic regression by hand: $\hat{y} = \sigma(Xw + b)$ with the binary cross-entropy loss. With PyTorch's tools:

* **`nn.Linear(30, 1)`** computes the score $z = Xw + b$, called the **logit**
* **`nn.BCEWithLogitsLoss()`** applies the sigmoid *and* computes the binary cross-entropy in one step, in a numerically stable way. So the model itself does **not** end with a sigmoid.
* **`torch.optim.SGD`** does the update

And the five-step loop is exactly the one from Notebook 2:

In [ ]:
torch.manual_seed(42)
model = nn.Linear(n_features, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    logits = model(X_train)              # 1. forward pass
    loss = loss_fn(logits, y_train)      # 2. loss
    optimizer.zero_grad()                # 3. clear old gradients
    loss.backward()                      # 4. backward pass
    optimizer.step()                     # 5. update
    if epoch % 20 == 0 or epoch == 99:
        print(f"epoch {epoch:3d}   loss {loss.item():.4f}")

## From logits to predictions

* `torch.sigmoid(logits)` turns logits into probabilities
* the predicted class is 1 when the probability is above 0.5, which is the same as **the logit being above 0**

Predictions don't need gradients, so we use `torch.no_grad()`.

In [ ]:
with torch.no_grad():
    logits = model(X_valid)

probabilities = torch.sigmoid(logits)
predictions = (logits > 0).float()

print("first 5 probabilities:", probabilities[:5].squeeze(1).round(decimals=3))
print("first 5 predictions:  ", predictions[:5].squeeze(1))
print("first 5 labels:       ", y_valid[:5].squeeze(1))
print(f"validation accuracy: {(predictions == y_valid).float().mean():.3f}")

# Part 2: Mini-batches with `Dataset` and `DataLoader`

The loop above uses the **whole** training set for every update (full-batch gradient descent). In Notebook 1 we wrote a `minibatches` generator that shuffled the indices and sliced out batches. PyTorch provides the same thing as two pieces:

* a **`Dataset`** knows how many examples there are and how to get example number `i`. `TensorDataset(X, y)` wraps tensors we already have.
* a **`DataLoader`** serves up batches from a dataset. With `shuffle=True`, it reshuffles at the start of every epoch.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
print("number of examples:", len(train_dataset))

x0, y0 = train_dataset[0]        # one example: (features, label)
print(x0.shape, y0)

In [ ]:
torch.manual_seed(42)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=256)   # no need to shuffle
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

X_batch, y_batch = next(iter(train_loader))
print("batches per epoch:", len(train_loader))
print("one batch:", X_batch.shape, y_batch.shape)

With 364 training examples and batches of 32, one epoch is 12 batches (the last one has only 12 examples). That's 12 updates per epoch instead of 1.

# Part 3: Reusable training and evaluation functions

We'll write the training loop once, as functions, and reuse it for every model from now on.

* **`train_one_epoch`** runs the five steps on every batch. It returns the average loss and the accuracy.
* **`evaluate`** computes the loss and accuracy on a dataset, inside `torch.no_grad()`.
* **`fit`** calls both once per epoch and records the results in a `history` dictionary.

Three habits from Notebook 2 are built in:
* **`model.train()` and `model.eval()`** switch the model between training and evaluation mode. Our models behave the same either way, but some layers (coming in later notebooks) don't.
* **Each batch is moved to `device`** inside the loop. The data itself stays on the CPU.
* **Each batch's loss is weighted by its size**, because the last batch can be smaller.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
        correct += ((logits > 0).float() == y_batch).sum().item()
        n += len(X_batch)
    return total_loss / n, correct / n


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(X_batch)
            correct += ((logits > 0).float() == y_batch).sum().item()
            n += len(X_batch)
    return total_loss / n, correct / n


def fit(model, train_loader, valid_loader, loss_fn, optimizer, n_epochs, verbose=True):
    history = {"train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": []}
    for epoch in range(n_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
        for key, value in zip(history, (train_loss, train_acc, valid_loss, valid_acc)):
            history[key].append(value)
        if verbose:
            print(f"epoch {epoch + 1:3d}   train loss {train_loss:.4f}  acc {train_acc:.3f}"
                  f"   |   valid loss {valid_loss:.4f}  acc {valid_acc:.3f}")
    return history

Let's use them on logistic regression, now with mini-batches. Remember the GPU rule: **move the model to the device first, then create the optimizer.**

In [ ]:
torch.manual_seed(42)
model = nn.Linear(n_features, 1).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
history_logistic = fit(model, train_loader, valid_loader, nn.BCEWithLogitsLoss(), optimizer, n_epochs=20)

In [ ]:
def plot_history(history, title=""):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(epochs, history["train_loss"], ".--", label="training")
    ax[0].plot(epochs, history["valid_loss"], ".-", label="validation")
    ax[0].set_ylabel("loss")
    ax[1].plot(epochs, history["train_acc"], ".--", label="training")
    ax[1].plot(epochs, history["valid_acc"], ".-", label="validation")
    ax[1].set_ylabel("accuracy")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=0.3); a.legend()
        a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

plot_history(history_logistic, "Logistic regression, mini-batch SGD")

# Part 4: A neural network with `nn.Sequential`

`nn.Sequential` chains layers together: the output of each layer is the input of the next. This network has two hidden layers (16 and 8 units, with ReLU activations), like the one we built by hand in Notebook 1, Part 4. It ends with **one output, the logit**, and no sigmoid.

In [ ]:
def make_mlp(seed=42):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(n_features, 16),
        nn.ReLU(),
        nn.Linear(16, 8),
        nn.ReLU(),
        nn.Linear(8, 1),
    ).to(device)

model = make_mlp()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

Training uses **exactly the same** `fit` function. Only the model changed.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
history_mlp = fit(model, train_loader, valid_loader, nn.BCEWithLogitsLoss(), optimizer, n_epochs=20)
plot_history(history_mlp, "Neural network, mini-batch SGD")

## Better optimizers: momentum and Adam

Plain SGD takes a step of size `lr × gradient` each time. Two common improvements:

* **Momentum** keeps a running average of recent gradients and moves in that direction, like a ball rolling downhill that builds up speed. This smooths out the noise from mini-batches. `torch.optim.SGD(..., momentum=0.9)`
* **Adam** adds momentum, plus an automatically adapted step size for **each** parameter, based on the size of its recent gradients. It works well with little tuning, so it's the most common default. `torch.optim.Adam(..., lr=0.001)`

Changing the optimizer is a one-line change:

In [ ]:
histories = {"SGD": history_mlp}

model = make_mlp()
histories["SGD + momentum"] = fit(model, train_loader, valid_loader, nn.BCEWithLogitsLoss(),
                                  torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9),
                                  n_epochs=20, verbose=False)

model = make_mlp()
histories["Adam"] = fit(model, train_loader, valid_loader, nn.BCEWithLogitsLoss(),
                        torch.optim.Adam(model.parameters(), lr=0.001),
                        n_epochs=20, verbose=False)

plt.figure(figsize=(6, 4))
for name, h in histories.items():
    plt.plot(range(1, 21), h["valid_loss"], ".-", label=name)
    print(f"{name:15s} final validation loss {h['valid_loss'][-1]:.4f}   accuracy {h['valid_acc'][-1]:.3f}")
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.grid(alpha=0.3); plt.legend()
plt.xticks(range(0, 21, 5))
plt.title("Validation loss by optimizer")
plt.show()

All three reach the same accuracy on this small, fairly easy dataset. Notice that plain SGD was actually the fastest here: its learning rate of 0.1 happens to suit this problem well. The "better" optimizers aren't better on every problem. Their advantage is that they work well on **many** problems with little tuning, which is why Adam is the usual starting point, and why we'll use it on the images in the next notebook.

# Part 5: Writing a model as a class with `nn.Module`

`nn.Sequential` works when data flows straight through one layer after another. Most PyTorch code you'll read instead defines each model as a **class**, which can describe any computation. We'll need this later.

* **`__init__`** runs once, when the model is created. That's where the layers are created and stored as attributes (`self.hidden1 = ...`).
* **`super().__init__()`** runs `nn.Module`'s own setup. Always call it first.
* **`forward(self, x)`** describes the computation. Calling `model(x)` runs `forward`.

PyTorch finds every layer stored in `__init__`, so `model.parameters()`, `.to(device)`, and the optimizer all work automatically.

In [ ]:
class ClassifierMLP(nn.Module):
    def __init__(self, n_inputs, n_hidden1=16, n_hidden2=8):
        super().__init__()
        self.hidden1 = nn.Linear(n_inputs, n_hidden1)
        self.hidden2 = nn.Linear(n_hidden1, n_hidden2)
        self.output = nn.Linear(n_hidden2, 1)

    def forward(self, x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        return self.output(x)          # the logit: no sigmoid

torch.manual_seed(42)
model = ClassifierMLP(n_features).to(device)
model

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
history_class = fit(model, train_loader, valid_loader, nn.BCEWithLogitsLoss(), optimizer,
                    n_epochs=30, verbose=False)
print(f"final validation accuracy: {history_class['valid_acc'][-1]:.3f}")

## Final check on the test set

Once we're done comparing models on the validation set, we evaluate the final model on the test set, once.

We also compare the accuracy with a baseline simplest possible "model": always predicting the most common class.

In [ ]:
from sklearn.metrics import confusion_matrix

test_loss, test_acc = evaluate(model, test_loader, nn.BCEWithLogitsLoss())

model.eval()
with torch.no_grad():
    test_pred = (model(X_test.to(device)) > 0).float().cpu()   # back to the CPU for scikit-learn

baseline = max(y_test.mean().item(), 1 - y_test.mean().item())
print(f"test accuracy: {test_acc:.3f}   (always predicting 'benign': {baseline:.3f})")
print("confusion matrix (rows = true class, columns = predicted):")
print(confusion_matrix(y_test.numpy(), test_pred.numpy()))

# Part 6: From 2 classes to 10

In the next notebook we classify handwritten digits: **10 classes** instead of 2. Here's what changes:

| | 2 classes (this notebook) | 10 classes (next notebook) |
|---|---|---|
| Model output | 1 logit | 10 logits, one per class |
| Turning logits into probabilities | sigmoid | **softmax** |
| Loss | `nn.BCEWithLogitsLoss` | **`nn.CrossEntropyLoss`** |
| Labels | floats, shape `(n, 1)` | **integers**, shape `(n,)` |
| Predicted class | `logit > 0` | `logits.argmax(dim=1)` |

**Softmax** turns a list of logits $z_1, \dots, z_K$ into probabilities that are positive and add up to 1:
$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

**Cross-entropy** is the negative log of the probability given to the correct class, $-\log p_{\text{correct}}$, just like binary cross-entropy. `nn.CrossEntropyLoss` takes the logits and the integer labels, and applies the softmax itself, so again **the model doesn't end with a softmax.**

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0],     # 2 examples, 3 classes
                       [0.1, 0.2,  3.0]])
labels = torch.tensor([0, 2])                # integer class labels

probs = torch.softmax(logits, dim=1)
print("probabilities:\n", probs.round(decimals=3))
print("each row sums to:", probs.sum(dim=1))
print("predicted classes:", logits.argmax(dim=1))

manual = -torch.log(probs[torch.arange(2), labels]).mean()
print("\n-log p[correct], averaged:", manual.item())
print("nn.CrossEntropyLoss:      ", nn.CrossEntropyLoss()(logits, labels).item())

With two classes, softmax over 2 logits and sigmoid over 1 logit are equivalent. The 10-class version is the same idea with more outputs.

# Summary

The complete workflow:

```python
# data: tensors -> Dataset -> DataLoader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

# model on the device, then the loss and the optimizer
model = nn.Sequential(nn.Linear(n_in, 16), nn.ReLU(), nn.Linear(16, 1)).to(device)
loss_fn = nn.BCEWithLogitsLoss()          # nn.MSELoss() for regression, nn.CrossEntropyLoss() for many classes
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# training: the five steps, for every batch, for several epochs
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        loss = loss_fn(model(X_batch), y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# evaluation
model.eval()
with torch.no_grad():
    predictions = model(X_test.to(device)).cpu()
```